# AU-FISH V3 Extension: YOLOv10n Multi-Seed Baseline

Notebook ini sepenuhnya terpisah dari V3 utama.

Parent V3 hanya dibaca:

```text
dataset_v2/experiments/multiseed_wbf_results_ready_v3_complete/
```

Seluruh output extension ditulis ke:

```text
dataset_v2/experiments/multiseed_wbf_results_ready_v3_extension_yolov10n_v1/
```

Scope:

- YOLOv10n saja
- time-based dan source-based
- seed 42, 123, 2026
- 100 epoch
- input 640, batch 16
- validation-selected `best.pt`
- common pycocotools COCOeval yang sama dengan V3
- mAP50-95, mAP50, mAP75, coral AP, fish AP
- optional FPS dan optional tabel gabungan V3

MambaCoral tidak dimasukkan karena pipeline reproduksibel resminya belum diberikan.

In [ ]:
#@title 1. Install
%pip -q install ultralytics==8.4.102 pycocotools pandas pyyaml pillow tqdm psutil

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 82.4 MB/s eta 0:00:00


In [ ]:
#@title 2. Mount Drive
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
#@title 3. Configuration, paths, and utilities
import zipfile

from pathlib import Path
from datetime import datetime
from typing import Any, Dict, List, Optional, Sequence
from collections import defaultdict
from contextlib import redirect_stdout
from io import StringIO

import os
import re
import gc
import json
import time
import random
import hashlib
import shutil
import subprocess
import platform
import importlib.metadata as importlib_metadata

import numpy as np
import pandas as pd
import yaml

V3_TAG = "multiseed_wbf_results_ready_v3_complete"
EXTENSION_TAG = "multiseed_wbf_results_ready_v3_extension_yolov10n_v1"

PROTOCOLS = ["time_based", "source_based"]
SPLITS = ["train", "val", "test"]
SEEDS = [42, 123, 2026]

MODEL_NAME = "yolov10n"
PRETRAINED_WEIGHTS = "yolov10n.pt"

TRAIN_CONFIG = {
    "epochs": 100,
    "imgsz": 640,
    "batch": 16,
    "workers": 0,
    "cache": False,
    "amp": True,
    "pretrained": True,
}

INFERENCE_CONFIG = {
    "conf": 0.001,
    "iou": 0.70,
    "max_det": 300,
    "imgsz": 640,
    "batch": 8,
    "augment": False,
}

RUN_TRAINING = True
RUN_EVALUATION = True
RUN_OPTIONAL_FPS = False
RUN_OPTIONAL_V3_COMPARISON = True

FPS_SEED = 42
FPS_IMAGES_PER_PROTOCOL = 100
FPS_WARMUP = 10
FPS_REPEATS = 3

PROJECT_ROOT_CANDIDATES = [
    Path("/content/drive/MyDrive/au-fish"),
    Path("/content/drive/.shortcut-targets-by-id/1kW1zeaSZ1KWylT60LjX9VMVZYNBF137Z/au-fish"),
]

def resolve_project_root() -> Path:
    for candidate in PROJECT_ROOT_CANDIDATES:
        if (candidate / "dataset_v2").is_dir():
            return candidate.resolve()

    for search_root in [
        Path("/content/drive/MyDrive"),
        Path("/content/drive/.shortcut-targets-by-id"),
    ]:
        if not search_root.exists():
            continue
        for dataset_root in search_root.rglob("dataset_v2"):
            if dataset_root.is_dir() and all(
                (dataset_root / protocol).is_dir() for protocol in PROTOCOLS
            ):
                return dataset_root.parent.resolve()

    raise FileNotFoundError("Folder induk AU-FISH tidak ditemukan.")

PROJECT_ROOT = resolve_project_root()
DATASET_ROOT = PROJECT_ROOT / "dataset_v2"

V3_ROOT = DATASET_ROOT / "experiments" / V3_TAG
V3_CANONICAL_ROOT = V3_ROOT / "canonical_data"
V3_MANIFESTS_ROOT = V3_ROOT / "manifests"
V3_REPORTS_ROOT = V3_ROOT / "reports"

EXTENSION_ROOT = DATASET_ROOT / "experiments" / EXTENSION_TAG
CONFIG_ROOT = EXTENSION_ROOT / "config"
RUNS_ROOT = EXTENSION_ROOT / "runs"
PREDICTIONS_ROOT = EXTENSION_ROOT / "predictions"
REPORTS_ROOT = EXTENSION_ROOT / "reports"
MANIFESTS_ROOT = EXTENSION_ROOT / "manifests"
LOGS_ROOT = EXTENSION_ROOT / "logs"

for folder in [
    CONFIG_ROOT,
    RUNS_ROOT,
    PREDICTIONS_ROOT,
    REPORTS_ROOT,
    MANIFESTS_ROOT,
    LOGS_ROOT,
]:
    folder.mkdir(parents=True, exist_ok=True)

LOCAL_CACHE_ROOT = Path("/content") / f"{EXTENSION_TAG}_cache"
LOCAL_RUNS_ROOT = Path("/content") / f"{EXTENSION_TAG}_runs"
LOCAL_CACHE_ROOT.mkdir(parents=True, exist_ok=True)
LOCAL_RUNS_ROOT.mkdir(parents=True, exist_ok=True)

def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()

def load_json(path: Path):
    with path.open("r", encoding="utf-8") as handle:
        return json.load(handle)

def atomic_json(value: Any, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    with temporary.open("w", encoding="utf-8") as handle:
        json.dump(value, handle, indent=2, ensure_ascii=False, default=str)
    temporary.replace(path)

def save_csv(dataframe: pd.DataFrame, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    dataframe.to_csv(path, index=False)

def sample_sd(values: Sequence[float]) -> float:
    array = np.asarray(values, dtype=float)
    return float(np.std(array, ddof=1)) if len(array) > 1 else float("nan")

scope_contract = {
    "extension_tag": EXTENSION_TAG,
    "parent_v3_tag": V3_TAG,
    "model": MODEL_NAME,
    "pretrained_weights": PRETRAINED_WEIGHTS,
    "protocols": PROTOCOLS,
    "seeds": SEEDS,
    "expected_training_runs": len(PROTOCOLS) * len(SEEDS),
    "parent_v3_read_only": True,
    "extension_output_root": str(EXTENSION_ROOT),
    "mambacoral_included": False,
}
atomic_json(scope_contract, CONFIG_ROOT / "extension_scope_contract.json")

print("V3 root:", V3_ROOT)
print("Extension root:", EXTENSION_ROOT)
print(json.dumps(scope_contract, indent=2))

V3 root: /content/drive/.shortcut-targets-by-id/1kW1zeaSZ1KWylT60LjX9VMVZYNBF137Z/au-fish/dataset_v2/experiments/multiseed_wbf_results_ready_v3_complete
Extension root: /content/drive/.shortcut-targets-by-id/1kW1zeaSZ1KWylT60LjX9VMVZYNBF137Z/au-fish/dataset_v2/experiments/multiseed_wbf_results_ready_v3_extension_yolov10n_v1
{
  "extension_tag": "multiseed_wbf_results_ready_v3_extension_yolov10n_v1",
  "parent_v3_tag": "multiseed_wbf_results_ready_v3_complete",
  "model": "yolov10n",
  "pretrained_weights": "yolov10n.pt",
  "protocols": [
    "time_based",
    "source_based"
  ],
  "seeds": [
    42,
    123,
    2026
  ],
  "expected_training_runs": 6,
  "parent_v3_read_only": true,
  "extension_output_root": "/content/drive/.shortcut-targets-by-id/1kW1zeaSZ1KWylT60LjX9VMVZYNBF137Z/au-fish/dataset_v2/experiments/multiseed_wbf_results_ready_v3_extension_yolov10n_v1",
  "mambacoral_included": false
}


In [ ]:
#@title 4. Read-only V3 canonical gate

lock_path = V3_MANIFESTS_ROOT / "canonical_dataset_lock.json"

if not lock_path.exists():
    raise FileNotFoundError(
        "V3 canonical lock tidak ditemukan. Jalankan V3 Notebook 00 sampai selesai."
    )

lock = load_json(lock_path)

if not lock.get("ready_for_training", False):
    raise RuntimeError("V3 canonical dataset belum terkunci untuk training.")

canonical_rows = []

for protocol in PROTOCOLS:
    for split in SPLITS:
        path = V3_CANONICAL_ROOT / protocol / split / "annotations.json"

        if not path.exists():
            raise FileNotFoundError(path)

        data = load_json(path)
        categories = {
            int(category["id"]): str(category["name"])
            for category in data.get("categories", [])
        }

        if categories != {1: "coral", 2: "fish"}:
            raise RuntimeError(f"Category mapping tidak sesuai: {path}")

        canonical_rows.append({
            "protocol": protocol,
            "split": split,
            "path": str(path),
            "sha256": sha256_file(path),
            "images": len(data.get("images", [])),
            "annotations": len(data.get("annotations", [])),
        })

canonical_reference_df = pd.DataFrame(canonical_rows)
save_csv(
    canonical_reference_df,
    MANIFESTS_ROOT / "v3_canonical_reference_manifest.csv",
)

atomic_json(
    {
        "parent_v3_lock_path": str(lock_path),
        "parent_v3_lock_sha256": sha256_file(lock_path),
        "parent_v3_read_only": True,
    },
    MANIFESTS_ROOT / "v3_read_only_contract.json",
)

display(canonical_reference_df)

,protocol,split,path,sha256,images,annotations
0,time_based,train,/content/drive/.shortcut-targets-by-id/1kW1zea...,ce8d95f2f25ede1c5fbdb21bed94e8e8158440a5d878fb...,1061,6465
1,time_based,val,/content/drive/.shortcut-targets-by-id/1kW1zea...,a361d742c1812bbb61ea7377e03ffb19e2db41020557f7...,203,913
2,time_based,test,/content/drive/.shortcut-targets-by-id/1kW1zea...,3a8bf5b2431f57615630b7228682428e4bb448d7bfb8f7...,282,1400
3,source_based,train,/content/drive/.shortcut-targets-by-id/1kW1zea...,02ea0cc3c72010b44f38ac6a09d8121e079e4d97308e8b...,1083,5101
4,source_based,val,/content/drive/.shortcut-targets-by-id/1kW1zea...,99d73ef8bba42c62819c9e21f23c17c787d01bb354e151...,232,1857
5,source_based,test,/content/drive/.shortcut-targets-by-id/1kW1zea...,d292fa8bc17ca12637b21059a38c195992765f7ce277ef...,231,1820


In [ ]:
#@title 5. Common COCOeval identical to V3

from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

COCO_MAX_DETS = [1, 10, 100]

def per_class_ap(evaluator: COCOeval) -> Dict[int, float]:
    precision = evaluator.eval.get("precision")
    if precision is None:
        return {}

    output = {}
    for category_index, category_id in enumerate(evaluator.params.catIds):
        values = precision[:, :, category_index, 0, -1]
        values = values[values > -1]
        output[int(category_id)] = (
            float(values.mean()) if values.size else float("nan")
        )

    return output

def evaluate_coco_predictions(
    gt_path: Path,
    predictions: List[Dict[str, Any]],
    temporary_name: str,
) -> Dict[str, Any]:
    temporary_path = Path("/content") / temporary_name
    atomic_json(predictions, temporary_path)

    coco_gt = COCO(str(gt_path))

    if not predictions:
        temporary_path.unlink(missing_ok=True)
        return {
            "map50_95": 0.0,
            "map50": 0.0,
            "map75": 0.0,
            "coral_ap": 0.0,
            "fish_ap": 0.0,
            "worst_class_ap": 0.0,
            "detections": 0,
            "coco_summary": "No predictions",
        }

    coco_dt = coco_gt.loadRes(str(temporary_path))
    evaluator = COCOeval(coco_gt, coco_dt, "bbox")
    evaluator.params.imgIds = sorted(coco_gt.getImgIds())
    evaluator.params.catIds = sorted(coco_gt.getCatIds())
    evaluator.params.maxDets = list(COCO_MAX_DETS)

    buffer = StringIO()
    with redirect_stdout(buffer):
        evaluator.evaluate()
        evaluator.accumulate()
        evaluator.summarize()

    aps = per_class_ap(evaluator)
    coral_ap = float(aps.get(1, float("nan")))
    fish_ap = float(aps.get(2, float("nan")))

    temporary_path.unlink(missing_ok=True)

    return {
        "map50_95": float(evaluator.stats[0]),
        "map50": float(evaluator.stats[1]),
        "map75": float(evaluator.stats[2]),
        "coral_ap": coral_ap,
        "fish_ap": fish_ap,
        "worst_class_ap": float(np.nanmin([coral_ap, fish_ap])),
        "detections": len(predictions),
        "coco_summary": buffer.getvalue(),
    }

In [ ]:
#@title 6. Local cache and YOLO label builder

from google.colab import drive

def remount_drive():
    try:
        drive.flush_and_unmount()
    except Exception:
        pass

    os.system("fusermount -u /content/drive 2>/dev/null")
    os.makedirs("/content/drive", exist_ok=True)
    drive.mount("/content/drive", force_remount=True)

def rsync_directory(source: Path, destination: Path, retries: int = 3):
    destination.mkdir(parents=True, exist_ok=True)
    command = [
        "rsync",
        "-a",
        "--partial",
        "--inplace",
        f"{source}/",
        f"{destination}/",
    ]

    last_error = None
    for attempt in range(1, retries + 1):
        try:
            subprocess.run(command, check=True)
            return
        except Exception as error:
            last_error = error
            print(f"rsync attempt {attempt} gagal:", repr(error))
            if attempt < retries:
                remount_drive()
                time.sleep(3)

    raise RuntimeError(f"Gagal menyalin {source}: {last_error}")

def relative_file_name(file_name: str) -> Path:
    parts = list(Path(str(file_name).replace("\\", "/")).parts)

    while parts and parts[0].lower() in {"train", "val", "test", "images"}:
        parts = parts[1:]

    if not parts:
        raise ValueError(f"file_name tidak valid: {file_name}")

    return Path(*parts)

def resolve_cached_image(image_root: Path, file_name: str) -> Path:
    relative = relative_file_name(file_name)

    for path in [
        image_root / relative,
        image_root / Path(file_name).name,
    ]:
        if path.exists():
            return path.resolve()

    matches = list(image_root.rglob(Path(file_name).name))

    if len(matches) == 1:
        return matches[0].resolve()

    if len(matches) > 1:
        raise RuntimeError(f"Ambiguous image basename: {file_name}")

    raise FileNotFoundError(file_name)

def build_local_protocol_cache(protocol: str) -> Dict[str, Path]:
    local_root = LOCAL_CACHE_ROOT / protocol

    for split in SPLITS:
        source_images = DATASET_ROOT / protocol / split / "images"
        local_images = local_root / split / "images"

        if not source_images.is_dir():
            raise FileNotFoundError(source_images)

        rsync_directory(source_images, local_images)

        source_gt = V3_CANONICAL_ROOT / protocol / split / "annotations.json"
        local_gt = local_root / split / "annotations.json"
        local_gt.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(source_gt, local_gt)

    return {"root": local_root, "yaml": local_root / f"{protocol}.yaml"}

def build_yolo_labels(local_root: Path, protocol: str) -> Path:
    for split in SPLITS:
        data = load_json(local_root / split / "annotations.json")
        images = {int(image["id"]): image for image in data["images"]}

        annotations_by_image = defaultdict(list)
        for annotation in data["annotations"]:
            annotations_by_image[int(annotation["image_id"])].append(annotation)

        image_root = local_root / split / "images"
        label_root = local_root / split / "labels"

        shutil.rmtree(label_root, ignore_errors=True)
        label_root.mkdir(parents=True, exist_ok=True)
        (local_root / split / "labels.cache").unlink(missing_ok=True)

        for image_id, image in images.items():
            image_path = resolve_cached_image(image_root, image["file_name"])
            relative = image_path.relative_to(image_root)
            label_path = (label_root / relative).with_suffix(".txt")
            label_path.parent.mkdir(parents=True, exist_ok=True)

            width = float(image["width"])
            height = float(image["height"])
            lines = []

            for annotation in annotations_by_image.get(image_id, []):
                category_id = int(annotation["category_id"])

                if category_id not in {1, 2}:
                    raise RuntimeError(f"Unknown category ID: {category_id}")

                class_index = category_id - 1
                x, y, w, h = [float(value) for value in annotation["bbox"]]

                x_center = (x + w / 2.0) / width
                y_center = (y + h / 2.0) / height
                normalized_width = w / width
                normalized_height = h / height

                lines.append(
                    f"{class_index} {x_center:.10f} {y_center:.10f} "
                    f"{normalized_width:.10f} {normalized_height:.10f}"
                )

            label_path.write_text("\n".join(lines), encoding="utf-8")

    yaml_path = local_root / f"{protocol}.yaml"

    with yaml_path.open("w", encoding="utf-8") as handle:
        yaml.safe_dump(
            {
                "path": str(local_root),
                "train": "train/images",
                "val": "val/images",
                "test": "test/images",
                "names": {0: "coral", 1: "fish"},
            },
            handle,
            sort_keys=False,
            allow_unicode=True,
        )

    return yaml_path

In [ ]:
#@title 7. Training helpers and environment manifest

import torch
from ultralytics import YOLO

DEVICE = 0 if torch.cuda.is_available() else "cpu"

def seed_everything(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def ultralytics_complete(run_dir: Path, expected_epochs: int) -> bool:
    best = run_dir / "weights" / "best.pt"
    last = run_dir / "weights" / "last.pt"
    results = run_dir / "results.csv"

    if not all([best.exists(), last.exists(), results.exists()]):
        return False

    try:
        return len(pd.read_csv(results)) >= expected_epochs
    except Exception:
        return False

def ultralytics_selected_epoch(run_dir: Path) -> Optional[int]:
    results_path = run_dir / "results.csv"

    if not results_path.exists():
        return None

    try:
        results = pd.read_csv(results_path)
    except Exception:
        return None

    if results.empty:
        return None

    results.columns = [str(column).strip() for column in results.columns]

    def normalize_column(column: str) -> str:
        return re.sub(
            r"[^a-z0-9]+",
            "_",
            str(column).strip().lower(),
        ).strip("_")

    candidates = []

    for column in results.columns:
        normalized_name = normalize_column(column)

        if "map50_95" in normalized_name:
            priority = 0
            if "metrics" in normalized_name:
                priority -= 2
            if normalized_name.endswith("_b") or "box" in normalized_name:
                priority -= 1
            candidates.append((priority, column))

    if not candidates:
        for column in results.columns:
            normalized_name = normalize_column(column)
            if "map50" in normalized_name and "95" in normalized_name:
                candidates.append((10, column))

    if not candidates:
        return None

    candidates.sort(key=lambda item: (item[0], item[1]))
    metric_column = candidates[0][1]
    metric = pd.to_numeric(results[metric_column], errors="coerce").dropna()

    if metric.empty:
        return None

    best_index = metric.idxmax()

    epoch_column = next(
        (
            column
            for column in results.columns
            if normalize_column(column) == "epoch"
        ),
        None,
    )

    if epoch_column is not None:
        epoch_value = pd.to_numeric(
            pd.Series([results.loc[best_index, epoch_column]]),
            errors="coerce",
        ).iloc[0]

        if pd.notna(epoch_value):
            return int(epoch_value) + 1

    return int(results.index.get_loc(best_index)) + 1

environment = {
    "extension_tag": EXTENSION_TAG,
    "model": MODEL_NAME,
    "python": platform.python_version(),
    "torch": torch.__version__,
    "ultralytics": importlib_metadata.version("ultralytics"),
    "pycocotools": importlib_metadata.version("pycocotools"),
    "cuda_available": torch.cuda.is_available(),
    "cuda_version": torch.version.cuda,
    "cudnn_version": torch.backends.cudnn.version(),
    "gpu": (
        torch.cuda.get_device_name(0)
        if torch.cuda.is_available()
        else None
    ),
    "gpu_memory_gb": (
        round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 3)
        if torch.cuda.is_available()
        else None
    ),
    "training_config": TRAIN_CONFIG,
    "inference_config": INFERENCE_CONFIG,
}

atomic_json(environment, MANIFESTS_ROOT / "extension_environment.json")
print(json.dumps(environment, indent=2))

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
{
  "extension_tag": "multiseed_wbf_results_ready_v3_extension_yolov10n_v1",
  "model": "yolov10n",
  "python": "3.12.13",
  "torch": "2.11.0+cu128",
  "ultralytics": "8.4.102",
  "pycocotools": "2.0.11",
  "cuda_available": true,
  "cuda_version": "12.8",
  "cudnn_version": 91900,
  "gpu": "NVIDIA L4",
  "gpu_memory_gb": 22.034,
  "training_config": {
    "epochs": 100,
    "imgsz": 640,
    "batch": 16,
    "workers": 0,
    "cache": false,
    "amp": true,
    "pretrained": true
  },
  "inference_config": {
    "conf": 0.001,
    "iou": 0.7,
    "max_det": 300,
    "imgsz": 640,
    "batch": 8,
    "augment": false
  }
}


In [ ]:
#@title 8. Train YOLOv10n for both protocols and all seeds

training_rows = []

if RUN_TRAINING:
    for protocol in PROTOCOLS:
        cache = build_local_protocol_cache(protocol)
        local_protocol_root = cache["root"]
        yolo_yaml = build_yolo_labels(local_protocol_root, protocol)

        for seed in SEEDS:
            seed_everything(seed)

            local_parent = (
                LOCAL_RUNS_ROOT / protocol / MODEL_NAME / f"seed_{seed}"
            )
            local_train = local_parent / "train"

            drive_train = (
                RUNS_ROOT / protocol / MODEL_NAME / f"seed_{seed}" / "train"
            )

            if ultralytics_complete(drive_train, TRAIN_CONFIG["epochs"]):
                best = drive_train / "weights" / "best.pt"
                training_rows.append({
                    "protocol": protocol,
                    "model": MODEL_NAME,
                    "seed": seed,
                    "status": "SKIP_COMPLETED",
                    "best_checkpoint": str(best),
                    "best_checkpoint_sha256": sha256_file(best),
                    "selected_epoch": ultralytics_selected_epoch(drive_train),
                })
                print("SKIP COMPLETED:", protocol, seed)
                continue

            if (
                drive_train.exists()
                and not (local_train / "weights" / "last.pt").exists()
            ):
                rsync_directory(drive_train, local_train)

            last_checkpoint = local_train / "weights" / "last.pt"

            def save_callback(trainer, destination=drive_train):
                try:
                    save_dir = Path(trainer.save_dir)
                    destination.mkdir(parents=True, exist_ok=True)

                    for relative in [
                        Path("weights/last.pt"),
                        Path("weights/best.pt"),
                        Path("results.csv"),
                        Path("args.yaml"),
                    ]:
                        source = save_dir / relative
                        target = destination / relative

                        if source.exists():
                            target.parent.mkdir(parents=True, exist_ok=True)
                            shutil.copy2(source, target)
                except Exception as error:
                    print("Checkpoint sync warning:", repr(error))

            if last_checkpoint.exists():
                print("RESUME:", protocol, seed)
                model = YOLO(str(last_checkpoint))
                model.add_callback("on_model_save", save_callback)
                model.train(resume=True)
            else:
                print("TRAIN FRESH:", protocol, seed)
                model = YOLO(PRETRAINED_WEIGHTS)
                model.add_callback("on_model_save", save_callback)
                model.train(
                    data=str(yolo_yaml),
                    epochs=TRAIN_CONFIG["epochs"],
                    imgsz=TRAIN_CONFIG["imgsz"],
                    batch=TRAIN_CONFIG["batch"],
                    seed=seed,
                    deterministic=True,
                    device=DEVICE,
                    workers=TRAIN_CONFIG["workers"],
                    cache=TRAIN_CONFIG["cache"],
                    amp=TRAIN_CONFIG["amp"],
                    pretrained=TRAIN_CONFIG["pretrained"],
                    project=str(local_parent),
                    name="train",
                    exist_ok=True,
                    save=True,
                    plots=True,
                    verbose=True,
                )

            rsync_directory(local_train, drive_train)

            if not ultralytics_complete(drive_train, TRAIN_CONFIG["epochs"]):
                raise RuntimeError(
                    f"Training tidak lengkap: {protocol}, seed {seed}"
                )

            best = drive_train / "weights" / "best.pt"

            training_rows.append({
                "protocol": protocol,
                "model": MODEL_NAME,
                "seed": seed,
                "status": "TRAINED",
                "epochs": TRAIN_CONFIG["epochs"],
                "best_checkpoint": str(best),
                "best_checkpoint_sha256": sha256_file(best),
                "selected_epoch": ultralytics_selected_epoch(drive_train),
                "gpu": environment["gpu"],
            })

            del model
            gc.collect()

            if torch.cuda.is_available():
                torch.cuda.empty_cache()

training_df = pd.DataFrame(training_rows)

if not training_df.empty:
    save_csv(training_df, REPORTS_ROOT / "yolov10n_training_summary.csv")
    display(training_df)

TRAIN FRESH: time_based 42
New https://pypi.org/project/ultralytics/8.4.112 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.102 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (NVIDIA L4, 22563MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/multiseed_wbf_results_ready_v3_extension_yolov10n_v1_cache/time_based/time_based.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_r

,protocol,model,seed,status,epochs,best_checkpoint,best_checkpoint_sha256,selected_epoch,gpu
0,time_based,yolov10n,42,TRAINED,100,/content/drive/.shortcut-targets-by-id/1kW1zea...,51728ec64c66001def89f4c7901ad09c822cdbc051829c...,85,NVIDIA L4
1,time_based,yolov10n,123,TRAINED,100,/content/drive/.shortcut-targets-by-id/1kW1zea...,49bfec835d3a3be485fab32468e9e3b1a1e751f264f8c6...,83,NVIDIA L4
2,time_based,yolov10n,2026,TRAINED,100,/content/drive/.shortcut-targets-by-id/1kW1zea...,214cb8b9c79586c85dfad5fb24590f5018532d4d5f5b15...,96,NVIDIA L4
3,source_based,yolov10n,42,TRAINED,100,/content/drive/.shortcut-targets-by-id/1kW1zea...,138cdca9dcb486c0626c5793f6388f55116034559263de...,86,NVIDIA L4
4,source_based,yolov10n,123,TRAINED,100,/content/drive/.shortcut-targets-by-id/1kW1zea...,1d1d7f41615e74541cf8a25a08964168635dd3144b4f3e...,53,NVIDIA L4
5,source_based,yolov10n,2026,TRAINED,100,/content/drive/.shortcut-targets-by-id/1kW1zea...,964b572dd8dcbc11256471de303189e5dd9456c3852d92...,33,NVIDIA L4


In [ ]:
#@title 9. Raw inference helpers

def canonical_image_records(local_root: Path, split: str):
    data = load_json(local_root / split / "annotations.json")
    return sorted(data["images"], key=lambda item: int(item["id"]))

def image_paths_for_split(local_root: Path, split: str):
    records = canonical_image_records(local_root, split)
    image_root = local_root / split / "images"
    paths = [
        resolve_cached_image(image_root, record["file_name"])
        for record in records
    ]
    return records, paths

def yolov10n_raw_predictions(
    model: YOLO,
    local_root: Path,
    split: str,
) -> List[Dict[str, Any]]:
    records, paths = image_paths_for_split(local_root, split)

    results = model.predict(
        source=[str(path) for path in paths],
        stream=True,
        batch=INFERENCE_CONFIG["batch"],
        imgsz=INFERENCE_CONFIG["imgsz"],
        conf=INFERENCE_CONFIG["conf"],
        iou=INFERENCE_CONFIG["iou"],
        max_det=INFERENCE_CONFIG["max_det"],
        augment=INFERENCE_CONFIG["augment"],
        verbose=False,
        device=DEVICE,
    )

    class_to_category = {"coral": 1, "fish": 2}
    predictions = []
    result_count = 0

    for result_count, result in enumerate(results, start=1):
        if result_count > len(records):
            raise RuntimeError("Ultralytics returned more results than inputs.")

        image_id = int(records[result_count - 1]["id"])

        if result.boxes is None:
            continue

        xyxy = result.boxes.xyxy.detach().cpu().numpy()
        scores = result.boxes.conf.detach().cpu().numpy()
        classes = result.boxes.cls.detach().cpu().numpy().astype(int)

        for box, score, class_index in zip(xyxy, scores, classes):
            class_name = str(model.names[int(class_index)]).strip().lower()

            if class_name not in class_to_category:
                raise RuntimeError(f"Unknown model class: {class_name}")

            x1, y1, x2, y2 = [float(value) for value in box]

            predictions.append({
                "image_id": image_id,
                "category_id": class_to_category[class_name],
                "bbox": [x1, y1, x2 - x1, y2 - y1],
                "score": float(score),
            })

    if result_count != len(records):
        raise RuntimeError(
            f"Prediction count mismatch: expected {len(records)}, got {result_count}"
        )

    return predictions

def prediction_manifest_matches(
    manifest_path: Path,
    protocol: str,
    seed: int,
    split: str,
    checkpoint_hash: str,
    gt_hash: str,
) -> bool:
    if not manifest_path.exists():
        return False

    try:
        manifest = load_json(manifest_path)
        return all([
            manifest.get("extension_tag") == EXTENSION_TAG,
            manifest.get("protocol") == protocol,
            manifest.get("model") == MODEL_NAME,
            int(manifest.get("seed")) == seed,
            manifest.get("split") == split,
            manifest.get("checkpoint_sha256") == checkpoint_hash,
            manifest.get("gt_sha256") == gt_hash,
            manifest.get("inference_config") == INFERENCE_CONFIG,
        ])
    except Exception:
        return False

In [ ]:
#@title 10. Generate val/test predictions and evaluate

result_rows = []
checkpoint_rows = []

if RUN_EVALUATION:
    for protocol in PROTOCOLS:
        cache = build_local_protocol_cache(protocol)
        local_protocol_root = cache["root"]

        for seed in SEEDS:
            drive_train = (
                RUNS_ROOT / protocol / MODEL_NAME / f"seed_{seed}" / "train"
            )
            checkpoint = drive_train / "weights" / "best.pt"

            if not checkpoint.exists():
                raise FileNotFoundError(checkpoint)

            checkpoint_hash = sha256_file(checkpoint)
            selected_epoch = ultralytics_selected_epoch(drive_train)

            checkpoint_rows.append({
                "protocol": protocol,
                "model": MODEL_NAME,
                "seed": seed,
                "checkpoint_path": str(checkpoint),
                "checkpoint_sha256": checkpoint_hash,
                "selected_epoch": selected_epoch,
            })

            model = YOLO(str(checkpoint))

            for split in ["val", "test"]:
                output_dir = (
                    PREDICTIONS_ROOT / protocol / MODEL_NAME / f"seed_{seed}"
                )
                output_dir.mkdir(parents=True, exist_ok=True)

                output_path = output_dir / f"{split}.json"
                manifest_path = output_dir / f"{split}_manifest.json"

                gt_path = (
                    V3_CANONICAL_ROOT / protocol / split / "annotations.json"
                )
                gt_hash = sha256_file(gt_path)

                reuse = (
                    output_path.exists()
                    and prediction_manifest_matches(
                        manifest_path,
                        protocol,
                        seed,
                        split,
                        checkpoint_hash,
                        gt_hash,
                    )
                )

                if reuse:
                    predictions = load_json(output_path)
                    status = "REUSED_VERIFIED"
                else:
                    predictions = yolov10n_raw_predictions(
                        model,
                        local_protocol_root,
                        split,
                    )
                    atomic_json(predictions, output_path)

                    atomic_json(
                        {
                            "extension_tag": EXTENSION_TAG,
                            "parent_v3_tag": V3_TAG,
                            "protocol": protocol,
                            "model": MODEL_NAME,
                            "seed": seed,
                            "split": split,
                            "checkpoint_path": str(checkpoint),
                            "checkpoint_sha256": checkpoint_hash,
                            "gt_path": str(gt_path),
                            "gt_sha256": gt_hash,
                            "inference_config": INFERENCE_CONFIG,
                            "prediction_path": str(output_path),
                            "prediction_sha256": sha256_file(output_path),
                        },
                        manifest_path,
                    )
                    status = "GENERATED"

                metrics = evaluate_coco_predictions(
                    gt_path,
                    predictions,
                    f"extension_{protocol}_{MODEL_NAME}_{seed}_{split}.json",
                )

                row = {
                    "extension_tag": EXTENSION_TAG,
                    "protocol": protocol,
                    "model": MODEL_NAME,
                    "seed": seed,
                    "split": split,
                    "status": status,
                    "selected_epoch": selected_epoch,
                    "checkpoint_path": str(checkpoint),
                    "checkpoint_sha256": checkpoint_hash,
                    "prediction_path": str(output_path),
                    "prediction_sha256": sha256_file(output_path),
                    "prediction_manifest_path": str(manifest_path),
                    "gt_path": str(gt_path),
                    "gt_sha256": gt_hash,
                }

                row.update({
                    key: value
                    for key, value in metrics.items()
                    if key != "coco_summary"
                })

                result_rows.append(row)

            del model
            gc.collect()

            if torch.cuda.is_available():
                torch.cuda.empty_cache()

checkpoint_manifest_df = pd.DataFrame(checkpoint_rows)
results_df = pd.DataFrame(result_rows)

if not checkpoint_manifest_df.empty:
    save_csv(
        checkpoint_manifest_df,
        MANIFESTS_ROOT / "yolov10n_checkpoint_manifest.csv",
    )

if not results_df.empty:
    save_csv(results_df, REPORTS_ROOT / "yolov10n_results_by_seed.csv")

    for protocol in PROTOCOLS:
        save_csv(
            results_df[results_df["protocol"] == protocol].copy(),
            REPORTS_ROOT / protocol / "individual_results_by_seed.csv",
        )

display(checkpoint_manifest_df)
display(results_df)

loading annotations into memory...
Done (t=0.01s)
creating index...
index created!
Loading and preparing results...
DONE (t=0.03s)
creating index...
index created!
loading annotations into memory...
Done (t=0.01s)
creating index...
index created!
Loading and preparing results...
DONE (t=0.05s)
creating index...
index created!
loading annotations into memory...
Done (t=0.01s)
creating index...
index created!
Loading and preparing results...
DONE (t=0.03s)
creating index...
index created!
loading annotations into memory...
Done (t=0.01s)
creating index...
index created!
Loading and preparing results...
DONE (t=0.05s)
creating index...
index created!
loading annotations into memory...
Done (t=0.01s)
creating index...
index created!
Loading and preparing results...
DONE (t=0.03s)
creating index...
index created!
loading annotations into memory...
Done (t=0.01s)
creating index...
index created!
Loading and preparing results...
DONE (t=0.04s)
creating index...
index created!
loading annotati

,protocol,model,seed,checkpoint_path,checkpoint_sha256,selected_epoch
0,time_based,yolov10n,42,/content/drive/.shortcut-targets-by-id/1kW1zea...,51728ec64c66001def89f4c7901ad09c822cdbc051829c...,85
1,time_based,yolov10n,123,/content/drive/.shortcut-targets-by-id/1kW1zea...,49bfec835d3a3be485fab32468e9e3b1a1e751f264f8c6...,83
2,time_based,yolov10n,2026,/content/drive/.shortcut-targets-by-id/1kW1zea...,214cb8b9c79586c85dfad5fb24590f5018532d4d5f5b15...,96
3,source_based,yolov10n,42,/content/drive/.shortcut-targets-by-id/1kW1zea...,138cdca9dcb486c0626c5793f6388f55116034559263de...,86
4,source_based,yolov10n,123,/content/drive/.shortcut-targets-by-id/1kW1zea...,1d1d7f41615e74541cf8a25a08964168635dd3144b4f3e...,53
5,source_based,yolov10n,2026,/content/drive/.shortcut-targets-by-id/1kW1zea...,964b572dd8dcbc11256471de303189e5dd9456c3852d92...,33


,extension_tag,protocol,model,seed,split,status,selected_epoch,checkpoint_path,checkpoint_sha256,prediction_path,...,prediction_manifest_path,gt_path,gt_sha256,map50_95,map50,map75,coral_ap,fish_ap,worst_class_ap,detections
0,multiseed_wbf_results_ready_v3_extension_yolov...,time_based,yolov10n,42,val,GENERATED,85,/content/drive/.shortcut-targets-by-id/1kW1zea...,51728ec64c66001def89f4c7901ad09c822cdbc051829c...,/content/drive/.shortcut-targets-by-id/1kW1zea...,...,/content/drive/.shortcut-targets-by-id/1kW1zea...,/content/drive/.shortcut-targets-by-id/1kW1zea...,a361d742c1812bbb61ea7377e03ffb19e2db41020557f7...,0.438615,0.667844,0.458175,0.579294,0.297936,0.297936,6476
1,multiseed_wbf_results_ready_v3_extension_yolov...,time_based,yolov10n,42,test,GENERATED,85,/content/drive/.shortcut-targets-by-id/1kW1zea...,51728ec64c66001def89f4c7901ad09c822cdbc051829c...,/content/drive/.shortcut-targets-by-id/1kW1zea...,...,/content/drive/.shortcut-targets-by-id/1kW1zea...,/content/drive/.shortcut-targets-by-id/1kW1zea...,3a8bf5b2431f57615630b7228682428e4bb448d7bfb8f7...,0.374915,0.580778,0.388373,0.502219,0.247612,0.247612,10454
2,multiseed_wbf_results_ready_v3_extension_yolov...,time_based,yolov10n,123,val,GENERATED,83,/content/drive/.shortcut-targets-by-id/1kW1zea...,49bfec835d3a3be485fab32468e9e3b1a1e751f264f8c6...,/content/drive/.shortcut-targets-by-id/1kW1zea...,...,/content/drive/.shortcut-targets-by-id/1kW1zea...,/content/drive/.shortcut-targets-by-id/1kW1zea...,a361d742c1812bbb61ea7377e03ffb19e2db41020557f7...,0.442041,0.662158,0.459140,0.589369,0.294712,0.294712,7168
3,multiseed_wbf_results_ready_v3_extension_yolov...,time_based,yolov10n,123,test,GENERATED,83,/content/drive/.shortcut-targets-by-id/1kW1zea...,49bfec835d3a3be485fab32468e9e3b1a1e751f264f8c6...,/content/drive/.shortcut-targets-by-id/1kW1zea...,...,/content/drive/.shortcut-targets-by-id/1kW1zea...,/content/drive/.shortcut-targets-by-id/1kW1zea...,3a8bf5b2431f57615630b7228682428e4bb448d7bfb8f7...,0.368149,0.569220,0.376762,0.490153,0.246145,0.246145,12120
4,multiseed_wbf_results_ready_v3_extension_yolov...,time_based,yolov10n,2026,val,GENERATED,96,/content/drive/.shortcut-targets-by-id/1kW1zea...,214cb8b9c79586c85dfad5fb24590f5018532d4d5f5b15...,/content/drive/.shortcut-targets-by-id/1kW1zea...,...,/content/drive/.shortcut-targets-by-id/1kW1zea...,/content/drive/.shortcut-targets-by-id/1kW1zea...,a361d742c1812bbb61ea7377e03ffb19e2db41020557f7...,0.440305,0.673686,0.453435,0.580632,0.299978,0.299978,6377
5,multiseed_wbf_results_ready_v3_extension_yolov...,time_based,yolov10n,2026,test,GENERATED,96,/content/drive/.shortcut-targets-by-id/1kW1zea...,214cb8b9c79586c85dfad5fb24590f5018532d4d5f5b15...,/content/drive/.shortcut-targets-by-id/1kW1zea...,...,/content/drive/.shortcut-targets-by-id/1kW1zea...,/content/drive/.shortcut-targets-by-id/1kW1zea...,3a8bf5b2431f57615630b7228682428e4bb448d7bfb8f7...,0.372714,0.578696,0.384310,0.503745,0.241682,0.241682,10020
6,multiseed_wbf_results_ready_v3_extension_yolov...,source_based,yolov10n,42,val,GENERATED,86,/content/drive/.shortcut-targets-by-id/1kW1zea...,138cdca9dcb486c0626c5793f6388f55116034559263de...,/content/drive/.shortcut-targets-by-id/1kW1zea...,...,/content/drive/.shortcut-targets-by-id/1kW1zea...,/content/drive/.shortcut-targets-by-id/1kW1zea...,99d73ef8bba42c62819c9e21f23c17c787d01bb354e151...,0.186601,0.316317,0.185163,0.276073,0.097129,0.097129,6733
7,multiseed_wbf_results_ready_v3_extension_yolov...,source_based,yolov10n,42,test,GENERATED,86,/content/drive/.shortcut-targets-by-id/1kW1zea...,138cdca9dcb486c0626c5793f6388f55116034559263de...,/content/drive/.shortcut-targets-by-id/1kW1zea...,...,/content/drive/.shortcut-targets-by-id/1kW1zea...,/content/drive/.shortcut-targets-by-id/1kW1zea...,d292fa8bc17ca12637b21059a38c195992765f7ce277ef...,0.164618,0.309500,0.151970,0.188030,0.141207,0.141207,11494
8,multiseed_wbf_results_ready_v3_extension_yolov...,source_based,yolov10n,123,val,GENERATED,53,/content/drive/.shortcu

In [ ]:
#@title 11. Summary and hard completion gate

results_path = REPORTS_ROOT / "yolov10n_results_by_seed.csv"
checkpoint_path = MANIFESTS_ROOT / "yolov10n_checkpoint_manifest.csv"

if not results_path.exists() or not checkpoint_path.exists():
    raise FileNotFoundError("Run training and evaluation first.")

results_df = pd.read_csv(results_path)
checkpoint_manifest_df = pd.read_csv(checkpoint_path)

metrics = [
    "map50_95",
    "map50",
    "map75",
    "coral_ap",
    "fish_ap",
    "worst_class_ap",
    "detections",
]

summary_rows = []

for keys, group in results_df.groupby(["protocol", "model", "split"]):
    protocol, model_name, split = keys
    row = {
        "protocol": protocol,
        "model": model_name,
        "split": split,
        "n_seeds": group["seed"].nunique(),
    }

    for metric in metrics:
        values = pd.to_numeric(group[metric], errors="coerce").dropna()
        row[f"{metric}_mean"] = float(values.mean())
        row[f"{metric}_sd"] = sample_sd(values.to_numpy(dtype=float))
        row[f"{metric}_min"] = float(values.min())
        row[f"{metric}_max"] = float(values.max())

    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows)
save_csv(summary_df, REPORTS_ROOT / "yolov10n_multiseed_summary.csv")

validation_df = results_df[results_df["split"] == "val"].copy()
test_df = results_df[results_df["split"] == "test"].copy()

expected = len(PROTOCOLS) * len(SEEDS)

completion = {
    "extension_tag": EXTENSION_TAG,
    "parent_v3_tag": V3_TAG,
    "model": MODEL_NAME,
    "expected_checkpoints": expected,
    "actual_checkpoints": len(checkpoint_manifest_df),
    "expected_validation_results": expected,
    "actual_validation_results": len(validation_df),
    "expected_test_results": expected,
    "actual_test_results": len(test_df),
    "checkpoint_keys_unique": (
        checkpoint_manifest_df[
            ["protocol", "model", "seed"]
        ].drop_duplicates().shape[0]
        == len(checkpoint_manifest_df)
    ),
    "validation_keys_unique": (
        validation_df[
            ["protocol", "model", "seed", "split"]
        ].drop_duplicates().shape[0]
        == len(validation_df)
    ),
    "test_keys_unique": (
        test_df[
            ["protocol", "model", "seed", "split"]
        ].drop_duplicates().shape[0]
        == len(test_df)
    ),
    "selected_epoch_complete": bool(
        pd.to_numeric(
            checkpoint_manifest_df["selected_epoch"],
            errors="coerce",
        ).notna().all()
    ),
    "checkpoint_files_present": bool(
        checkpoint_manifest_df["checkpoint_path"]
        .map(lambda value: Path(value).exists())
        .all()
    ),
    "prediction_files_present": bool(
        results_df["prediction_path"]
        .map(lambda value: Path(value).exists())
        .all()
    ),
    "exact_protocols": (
        set(checkpoint_manifest_df["protocol"].astype(str))
        == set(PROTOCOLS)
    ),
    "exact_seeds": (
        set(checkpoint_manifest_df["seed"].astype(int))
        == set(SEEDS)
    ),
    "writes_to_parent_v3": False,
}

completion["ready_for_reporting"] = all([
    completion["actual_checkpoints"] == completion["expected_checkpoints"],
    completion["actual_validation_results"] == completion["expected_validation_results"],
    completion["actual_test_results"] == completion["expected_test_results"],
    completion["checkpoint_keys_unique"],
    completion["validation_keys_unique"],
    completion["test_keys_unique"],
    completion["selected_epoch_complete"],
    completion["checkpoint_files_present"],
    completion["prediction_files_present"],
    completion["exact_protocols"],
    completion["exact_seeds"],
])

atomic_json(completion, MANIFESTS_ROOT / "extension_completion.json")

display(summary_df)
print(json.dumps(completion, indent=2))

if not completion["ready_for_reporting"]:
    raise RuntimeError("YOLOv10n extension belum lengkap.")

,protocol,model,split,n_seeds,map50_95_mean,map50_95_sd,map50_95_min,map50_95_max,map50_mean,map50_sd,...,fish_ap_min,fish_ap_max,worst_class_ap_mean,worst_class_ap_sd,worst_class_ap_min,worst_class_ap_max,detections_mean,detections_sd,detections_min,detections_max
0,source_based,yolov10n,test,3,0.157727,0.018333,0.136948,0.171616,0.306656,0.026471,...,0.106839,0.141207,0.127878,0.018435,0.106839,0.141207,17160.000000,5150.730337,11494.0,21559.0
1,source_based,yolov10n,val,3,0.195163,0.007809,0.186601,0.201895,0.341074,0.029671,...,0.090053,0.098405,0.095196,0.004499,0.090053,0.098405,14069.333333,6651.119931,6733.0,19705.0
2,time_based,yolov10n,test,3,0.371926,0.003451,0.368149,0.374915,0.576231,0.006160,...,0.241682,0.247612,0.245146,0.003088,0.241682,0.247612,10864.666667,1108.596109,10020.0,12120.0
3,time_based,yolov10n,val,3,0.440320,0.001713,0.438615,0.442041,0.667896,0.005764,...,0.294712,0.299978,0.297542,0.002655,0.294712,0.299978,6673.666667,430.957461,6377.0,7168.0


{
  "extension_tag": "multiseed_wbf_results_ready_v3_extension_yolov10n_v1",
  "parent_v3_tag": "multiseed_wbf_results_ready_v3_complete",
  "model": "yolov10n",
  "expected_checkpoints": 6,
  "actual_checkpoints": 6,
  "expected_validation_results": 6,
  "actual_validation_results": 6,
  "expected_test_results": 6,
  "actual_test_results": 6,
  "checkpoint_keys_unique": true,
  "validation_keys_unique": true,
  "test_keys_unique": true,
  "selected_epoch_complete": true,
  "checkpoint_files_present": true,
  "prediction_files_present": true,
  "exact_protocols": true,
  "exact_seeds": true,
  "writes_to_parent_v3": false,
  "ready_for_reporting": true
}


In [ ]:
#@title 12. Optional FPS on one common benchmark GPU

from PIL import Image

def synchronize():
    if torch.cuda.is_available():
        torch.cuda.synchronize()

def evenly_spaced_indices(total: int, desired: int) -> List[int]:
    if total <= desired:
        return list(range(total))

    return sorted(
        set(
            np.linspace(
                0,
                total - 1,
                num=desired,
                dtype=int,
            ).tolist()
        )
    )

fps_rows = []

if RUN_OPTIONAL_FPS:
    fps_environment = {
        "extension_tag": EXTENSION_TAG,
        "gpu": (
            torch.cuda.get_device_name(0)
            if torch.cuda.is_available()
            else None
        ),
        "torch": torch.__version__,
        "ultralytics": importlib_metadata.version("ultralytics"),
        "seed": FPS_SEED,
        "images_per_protocol": FPS_IMAGES_PER_PROTOCOL,
        "warmup": FPS_WARMUP,
        "repeats": FPS_REPEATS,
        "batch_size": 1,
        "input_size": TRAIN_CONFIG["imgsz"],
        "disk_io_in_timing": False,
    }
    atomic_json(fps_environment, MANIFESTS_ROOT / "fps_environment.json")

    for protocol in PROTOCOLS:
        cache = build_local_protocol_cache(protocol)
        local_root = cache["root"]
        _, paths = image_paths_for_split(local_root, "test")

        indices = evenly_spaced_indices(
            len(paths),
            FPS_IMAGES_PER_PROTOCOL,
        )

        arrays = []
        for index in indices:
            with Image.open(paths[index]) as image:
                arrays.append(np.asarray(image.convert("RGB")).copy())

        checkpoint = (
            RUNS_ROOT
            / protocol
            / MODEL_NAME
            / f"seed_{FPS_SEED}"
            / "train"
            / "weights"
            / "best.pt"
        )

        model = YOLO(str(checkpoint))

        for array in arrays[: min(FPS_WARMUP, len(arrays))]:
            _ = model.predict(
                source=array,
                imgsz=TRAIN_CONFIG["imgsz"],
                conf=INFERENCE_CONFIG["conf"],
                iou=INFERENCE_CONFIG["iou"],
                max_det=INFERENCE_CONFIG["max_det"],
                augment=False,
                verbose=False,
                device=DEVICE,
            )

        synchronize()

        for repeat in range(1, FPS_REPEATS + 1):
            latencies = []

            for array in arrays:
                synchronize()
                start = time.perf_counter()

                _ = model.predict(
                    source=array,
                    imgsz=TRAIN_CONFIG["imgsz"],
                    conf=INFERENCE_CONFIG["conf"],
                    iou=INFERENCE_CONFIG["iou"],
                    max_det=INFERENCE_CONFIG["max_det"],
                    augment=False,
                    verbose=False,
                    device=DEVICE,
                )

                synchronize()
                latencies.append(time.perf_counter() - start)

            latency_array = np.asarray(latencies, dtype=float)
            latency_ms = latency_array * 1000.0

            fps_rows.append({
                "protocol": protocol,
                "model": MODEL_NAME,
                "seed": FPS_SEED,
                "repeat": repeat,
                "images": len(arrays),
                "fps": float(1.0 / latency_array.mean()),
                "mean_latency_ms": float(latency_ms.mean()),
                "median_latency_ms": float(np.median(latency_ms)),
                "p95_latency_ms": float(np.percentile(latency_ms, 95)),
                "parameter_count": int(
                    sum(parameter.numel() for parameter in model.model.parameters())
                ),
                "checkpoint_path": str(checkpoint),
                "checkpoint_sha256": sha256_file(checkpoint),
            })

        del model
        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

fps_df = pd.DataFrame(fps_rows)

if not fps_df.empty:
    save_csv(
        fps_df,
        REPORTS_ROOT / "yolov10n_fps_repeated_results.csv",
    )

    fps_summary_rows = []

    for (protocol, model_name), group in fps_df.groupby(
        ["protocol", "model"]
    ):
        fps_summary_rows.append({
            "protocol": protocol,
            "model": model_name,
            "repeats": len(group),
            "fps_mean": float(group["fps"].mean()),
            "fps_sd": sample_sd(group["fps"].to_numpy(dtype=float)),
            "mean_latency_ms": float(group["mean_latency_ms"].mean()),
            "p95_latency_ms": float(group["p95_latency_ms"].mean()),
            "parameter_count": int(group["parameter_count"].iloc[0]),
        })

    fps_summary_df = pd.DataFrame(fps_summary_rows)
    save_csv(
        fps_summary_df,
        REPORTS_ROOT / "yolov10n_fps_summary.csv",
    )
    display(fps_summary_df)
else:
    print(
        "FPS skipped. Set RUN_OPTIONAL_FPS = True and run this cell "
        "on the same GPU used for the V3 FPS benchmark."
    )

FPS skipped. Set RUN_OPTIONAL_FPS = True and run this cell on the same GPU used for the V3 FPS benchmark.


In [ ]:
#@title 13. Optional read-only V3 comparison

if RUN_OPTIONAL_V3_COMPARISON:
    extension_path = REPORTS_ROOT / "yolov10n_results_by_seed.csv"

    if not extension_path.exists():
        print("Extension results belum tersedia.")
    else:
        frames = []

        extension_df = pd.read_csv(extension_path)
        extension_df["result_source"] = "extension_yolov10n"
        frames.append(extension_df)

        for protocol in PROTOCOLS:
            v3_path = (
                V3_REPORTS_ROOT
                / protocol
                / "individual_results_by_seed.csv"
            )

            if v3_path.exists():
                v3_df = pd.read_csv(v3_path)
                v3_df["result_source"] = "v3_core"
                frames.append(v3_df)

        comparison_df = pd.concat(frames, ignore_index=True, sort=False)
        save_csv(
            comparison_df,
            REPORTS_ROOT / "v3_plus_yolov10n_results_by_seed.csv",
        )

        test_df = comparison_df[comparison_df["split"] == "test"].copy()
        summary_rows = []

        for (protocol, model_name), group in test_df.groupby(
            ["protocol", "model"]
        ):
            summary_rows.append({
                "protocol": protocol,
                "model": model_name,
                "n_seeds": group["seed"].nunique(),
                "map50_95_mean": float(group["map50_95"].mean()),
                "map50_95_sd": sample_sd(
                    group["map50_95"].to_numpy(dtype=float)
                ),
                "map50_mean": float(group["map50"].mean()),
                "map75_mean": float(group["map75"].mean()),
                "coral_ap_mean": float(group["coral_ap"].mean()),
                "fish_ap_mean": float(group["fish_ap"].mean()),
                "worst_class_ap_mean": float(
                    group["worst_class_ap"].mean()
                ),
            })

        comparison_summary_df = (
            pd.DataFrame(summary_rows)
            .sort_values(
                ["protocol", "map50_95_mean"],
                ascending=[True, False],
            )
        )

        save_csv(
            comparison_summary_df,
            REPORTS_ROOT / "v3_plus_yolov10n_test_summary.csv",
        )
        display(comparison_summary_df)

,protocol,model,n_seeds,map50_95_mean,map50_95_sd,map50_mean,map75_mean,coral_ap_mean,fish_ap_mean,worst_class_ap_mean
1,source_based,rtdetr_l,3,0.278634,0.028003,0.463843,0.277130,0.318408,0.238859,0.238859
2,source_based,yolo11m,3,0.211547,0.012772,0.379708,0.203605,0.240357,0.182736,0.182736
0,source_based,faster_rcnn,3,0.209772,0.005072,0.392212,0.205151,0.281086,0.138457,0.138457
3,source_based,yolo11n,3,0.164212,0.019329,0.315533,0.146747,0.188809,0.139616,0.139616
4,source_based,yolov10n,3,0.157727,0.018333,0.306656,0.139690,0.187577,0.127878,0.127878
6,time_based,rtdetr_l,3,0.442618,0.009834,0.651454,0.469175,0.538584,0.346651,0.346651
7,time_based,yolo11m,3,0.412694,0.002741,0.635333,0.429275,0.497366,0.328022,0.328022
8,time_based,yolo11n,3,0.388341,0.004846,0.587285,0.402669,0.517036,0.259646,0.259646
9,time_based,yolov10n,3,0.371926,0.003451,0.576231,0.383148,0.498706,0.245146,0.245146
5,time_based,faster_rcnn,3,0.306293,0.014076,0.528752,0.308218,0.422216,0.190370,0.190370


In [ ]:
#@title 14. Final report and evidence package

completion_path = MANIFESTS_ROOT / "extension_completion.json"

if not completion_path.exists():
    raise FileNotFoundError(completion_path)

completion = load_json(completion_path)
summary_df = pd.read_csv(
    REPORTS_ROOT / "yolov10n_multiseed_summary.csv"
)

test_summary_df = summary_df[summary_df["split"] == "test"].copy()

lines = [
    "# AU-FISH YOLOv10n Extension",
    "",
    f"Extension tag: `{EXTENSION_TAG}`",
    "",
    f"Parent V3 tag: `{V3_TAG}`",
    "",
    f"Ready for reporting: **{completion['ready_for_reporting']}**",
    "",
    "## Scope",
    "",
    "- Model: YOLOv10n",
    "- Pretrained checkpoint: yolov10n.pt",
    "- Protocols: time-based and source-based",
    "- Seeds: 42, 123, 2026",
    "- Epochs: 100",
    "- Input size: 640",
    "- Batch size: 16",
    "- Parent V3 is read-only",
    "",
    "## Test summary",
    "",
]

for row in test_summary_df.itertuples():
    lines.append(
        f"- {row.protocol}: mAP50-95 {row.map50_95_mean:.6f} "
        f"± {row.map50_95_sd:.6f}; coral AP "
        f"{row.coral_ap_mean:.6f}; fish AP {row.fish_ap_mean:.6f}."
    )

lines.extend([
    "",
    "## Reporting boundary",
    "",
    "- YOLOv10n is an extension baseline, not a silent change to V3.",
    "- MambaCoral is not implemented without its exact reproducible pipeline.",
    "- FPS must be compared only under the same device and timing boundary.",
])

report_text = "\n".join(lines)
report_path = REPORTS_ROOT / "YOLOV10N_EXTENSION_REPORT.md"
report_path.write_text(report_text, encoding="utf-8")

evidence_zip = EXTENSION_ROOT / "YOLOV10N_EXTENSION_EVIDENCE.zip"
evidence_zip.unlink(missing_ok=True)

with zipfile.ZipFile(
    evidence_zip,
    mode="w",
    compression=zipfile.ZIP_DEFLATED,
) as archive:
    for source_root in [
        CONFIG_ROOT,
        MANIFESTS_ROOT,
        PREDICTIONS_ROOT,
        REPORTS_ROOT,
    ]:
        if not source_root.exists():
            continue

        for path in source_root.rglob("*"):
            if path.is_file():
                archive.write(
                    path,
                    arcname=str(path.relative_to(EXTENSION_ROOT)),
                )

print(report_text)
print("\nEvidence ZIP:", evidence_zip)

if not completion["ready_for_reporting"]:
    raise RuntimeError("Extension belum siap dilaporkan.")

# AU-FISH YOLOv10n Extension

Extension tag: `multiseed_wbf_results_ready_v3_extension_yolov10n_v1`

Parent V3 tag: `multiseed_wbf_results_ready_v3_complete`

Ready for reporting: **True**

## Scope

- Model: YOLOv10n
- Pretrained checkpoint: yolov10n.pt
- Protocols: time-based and source-based
- Seeds: 42, 123, 2026
- Epochs: 100
- Input size: 640
- Batch size: 16
- Parent V3 is read-only

## Test summary

- source_based: mAP50-95 0.157727 ± 0.018333; coral AP 0.187577; fish AP 0.127878.
- time_based: mAP50-95 0.371926 ± 0.003451; coral AP 0.498706; fish AP 0.245146.

## Reporting boundary

- YOLOv10n is an extension baseline, not a silent change to V3.
- MambaCoral is not implemented without its exact reproducible pipeline.
- FPS must be compared only under the same device and timing boundary.

Evidence ZIP: /content/drive/.shortcut-targets-by-id/1kW1zeaSZ1KWylT60LjX9VMVZYNBF137Z/au-fish/dataset_v2/experiments/multiseed_wbf_results_ready_v3_extension_yolov10n_v1/YOLOV10N_EXTENSION_EV